# RAG over summaries — the collapsed tree joins the index

Node summaries become retrievable exactly like source content: split into paragraphs by
the SAME structure-aware splitter (they're markdown — the grammar applies), stored in the
SAME `paragraphs`/`contents` tables under `source_type='node'`
(`node://hrono/nodes/<id>/<title-slug>@<sig>` — content model, opaque version = the
node's member signature), embedded into the same `(hash, model)` table, indexed by the
same FTS5 — then the FROZEN retrieval recipe runs unchanged over the union.

Hypothesis: architectural questions should now inject summary paragraphs (which know
cross-cutting facts no single chunk states), while detail questions keep hitting source
paragraphs. The old prompts are rerun to check for regressions, plus new architectural
prompts. Node paths carry the node TITLE slug so the α=0.5 path-blend has semantics to
match ("nodes/47/webphone-client-react-frontend", not "nodes/47").

In [1]:
import json
import re
import sqlite3
import tempfile
from hashlib import sha256
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer

from structuring import split_file_hier, sp_render

APP_DIR = Path.cwd()
DB_PATH = APP_DIR / ".hrono" / "hrono-v2.db"
MODEL_NAME = "ibm-granite/granite-embedding-97m-multilingual-r2"
BATCH = 16

db = sqlite3.connect(DB_PATH)
model = SentenceTransformer(MODEL_NAME)


def slug(title):
    return re.sub(r"[^a-z0-9]+", "-", title.lower()).strip("-")[:60]


# drop-and-rebuild the node-source rows (idempotent re-runs; source rows untouched)
with db:
    db.execute("DELETE FROM paragraphs WHERE source_type = 'node'")

nodes = db.execute("SELECT id, level, sig, summary FROM nodes WHERE summary IS NOT NULL").fetchall()
n_paras = 0
for nid, level, sig, summary in nodes:
    title = summary.lstrip("# ").split("\n")[0]
    path = f"nodes/{nid}/{slug(title)}"
    with tempfile.NamedTemporaryFile(suffix=".md", mode="w", delete=False) as f:
        f.write(summary)
        tmp = Path(f.name)
    _, chunks_ = split_file_hier(tmp)
    raw = tmp.read_bytes()
    tmp.unlink()
    with db:
        for chunk_no, c in enumerate(chunks_):
            for p in c.paragraphs:
                body = raw[p.byte_start:p.byte_end]
                h = sha256(body).hexdigest()
                db.execute("INSERT OR IGNORE INTO contents (hash, body) VALUES (?,?)", (h, body))
                db.execute(
                    "INSERT INTO paragraphs (source_type, instance, path, version, in_chunk,"
                    " byte_start, byte_end, hash, struct_paths, struct_uris)"
                    " VALUES ('node', 'hrono', ?, ?, ?, ?, ?, ?, ?, ?)",
                    (path, sig, chunk_no, p.byte_start, p.byte_end, h,
                     json.dumps(p.struct_paths),
                     " ".join(sp_render(sp) for sp in p.struct_paths)))
                n_paras += 1
print(f"{len(nodes)} summaries -> {n_paras} node paragraphs in the shared tables")

Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

48 summaries -> 610 node paragraphs in the shared tables


In [2]:
# embed whatever is new (content-addressed: unchanged summaries cost nothing)
body_of = dict(db.execute(
    "SELECT c.hash, c.body FROM contents c JOIN paragraphs p ON p.hash = c.hash GROUP BY c.hash"))
done = {r[0] for r in db.execute("SELECT hash FROM embeddings WHERE model = ?", (MODEL_NAME,))}
todo = [h for h in body_of if h not in done]
print(f"{len(todo)} new bodies to embed")
for i in range(0, len(todo), BATCH):
    batch = todo[i:i + BATCH]
    vecs = model.encode([body_of[h].decode('utf-8', 'replace') for h in batch],
                        batch_size=BATCH, normalize_embeddings=True, show_progress_bar=False)
    with db:
        db.executemany("INSERT INTO embeddings (hash, model, dims, vector) VALUES (?,?,?,?)",
                       [(h, MODEL_NAME, vecs.shape[1], np.asarray(v, np.float32).tobytes())
                        for h, v in zip(batch, vecs)])

# load the UNION retrieval space: source + node paragraphs, path-blended a=0.5
rows = db.execute("""
    SELECT p.hash, p.path, p.source_type, p.uri, p.struct_uris, e.vector
    FROM paragraphs p JOIN embeddings e ON e.hash = p.hash AND e.model = ?
    GROUP BY p.hash""", (MODEL_NAME,)).fetchall()
p_hashes = [r[0] for r in rows]
p_kind = {r[0]: r[2] for r in rows}
header_by_hash = {r[0]: f"{r[3]} {r[4]}".strip() for r in rows}
P = np.frombuffer(b"".join(r[5] for r in rows), np.float32).reshape(len(rows), -1)

paths = [r[1] for r in rows]
distinct = sorted(set(paths))
pv = dict(zip(distinct, model.encode(distinct, batch_size=BATCH,
                                     normalize_embeddings=True, show_progress_bar=False)))
PATH = np.stack([pv[p] for p in paths])
ALPHA = 0.5
M_BLEND = np.hstack([np.sqrt(1 - ALPHA) * P, np.sqrt(ALPHA) * PATH])

# BM25 over the union
db.executescript("DROP TABLE IF EXISTS para_fts; CREATE VIRTUAL TABLE para_fts USING fts5(text);")
with db:
    db.executemany("INSERT INTO para_fts (rowid, text) VALUES (?,?)",
                   [(pid, f"{path} {body.decode('utf-8', 'replace')}")
                    for pid, path, body in db.execute(
                        "SELECT p.id, p.path, c.body FROM paragraphs p"
                        " JOIN contents c ON c.hash = p.hash")])
hash_by_id = dict(db.execute("SELECT id, hash FROM paragraphs"))
print(f"union space: {len(p_hashes)} paragraphs "
      f"({sum(1 for h in p_hashes if p_kind[h] == 'node')} from summaries)")

0 new bodies to embed


union space: 4508 paragraphs (610 from summaries)


## The frozen recipe over the union — old prompts + architectural prompts

Recipe unchanged: Z_VEC 3.5 solo, keyword z ≥ 6 with vector support ≥ 3, weighted RRF
70/30, cap 10, empty → traverse. Hits tagged `[SUMMARY]` / `[source]`.

In [3]:
Z_VEC, Z_KW, Z_KW_SUPPORT, RRF_K, MAX_K = 3.5, 6.0, 3.0, 60, 10

PROMPTS = [
    # architectural — where summaries should win
    "What is the overall architecture of the webphone application?",
    "What conventions do the webphone React components follow across the codebase?",
    "What are the known cross-cutting hazards and risky patterns in the webphone frontend?",
    "How is logging implemented across the webphone server?",
    "What technologies does the football tournament project use end to end?",
    # detail — source paragraphs should keep winning (regression check)
    "What does ContactService.getByUser return and how is pagination handled?",
    "Which Spring Boot starter dependencies does the football match project use?",
    "How do I run the football match application locally?",
    "How does the webphone client translate its UI into Ukrainian?",
    "Under which license is webphone distributed?",
    "Where is the photo of a contact validated for maximum size?",
    # no-answer controls
    "How is payment processing implemented and which payment provider API is used?",
    "Where is the Kafka topic configuration and how are events consumed?",
]


import pandas as pd
from IPython.display import display, Markdown
pd.set_option("display.max_colwidth", 200)


def short(header):
    return re.sub(r'@[0-9a-f]{40,}', lambda m: m.group(0)[:9], header)


for prompt in PROMPTS:
    q = model.encode([prompt], normalize_embeddings=True)[0]
    qa = np.concatenate([np.sqrt(1 - ALPHA) * q, np.sqrt(ALPHA) * q])
    sims = M_BLEND @ qa
    v_mu, v_sd = float(sims.mean()), float(sims.std())
    order = np.argsort(-sims)
    v_rank = [p_hashes[i] for i in order[:60]]
    sim_by_hash = {p_hashes[i]: float(sims[i]) for i in order[:200]}

    terms = re.findall(r"[A-Za-z0-9_]+", prompt)
    rows = db.execute("SELECT rowid, bm25(para_fts) FROM para_fts WHERE para_fts MATCH ?"
                      " ORDER BY bm25(para_fts) LIMIT 2000", (" OR ".join(terms),)).fetchall()
    best = {}
    for rid, s_ in rows:
        best.setdefault(hash_by_id[rid], -s_)
    ks = np.array(list(best.values())) if best else np.array([0.0])
    k_mu, k_sd = float(ks.mean()), float(ks.std())
    k_rank = list(best)[:60]

    z_v = lambda h: (sim_by_hash[h] - v_mu) / v_sd if h in sim_by_hash else None
    z_k = lambda h: (best[h] - k_mu) / k_sd if h in best and k_sd > 0 else None
    v_qual = {h for h in v_rank[:MAX_K] if z_v(h) is not None and z_v(h) >= Z_VEC}
    k_qual = {h for h in k_rank[:MAX_K]
              if z_k(h) is not None and z_k(h) >= Z_KW
              and z_v(h) is not None and z_v(h) >= Z_KW_SUPPORT}

    def rrf(h):
        s_ = 0.0
        if h in v_rank:
            s_ += 0.7 / (RRF_K + v_rank.index(h) + 1)
        if h in k_rank:
            s_ += 0.3 / (RRF_K + k_rank.index(h) + 1)
        return s_

    final = sorted(v_qual | k_qual, key=rrf, reverse=True)[:MAX_K]
    candidates = sorted(set(v_rank[:MAX_K]) | set(k_rank[:MAX_K]), key=rrf, reverse=True)

    rows_out = []
    for h in candidates:
        sv, zv, sk, zk = sim_by_hash.get(h), z_v(h), best.get(h), z_k(h)
        uri, _, sp = short(header_by_hash[h]).partition(" ")
        rows_out.append({
            "kind": "SUMMARY" if p_kind[h] == "node" else "source",
            "cosine": None if sv is None else round(sv, 3),
            "z_vec": None if zv is None else round(zv, 2),
            "vQ": "Y" if h in v_qual else "",
            "bm25": None if sk is None else round(sk, 1),
            "z_kw": None if zk is None else round(zk, 2),
            "kQ": "Y" if h in k_qual else "",
            "rrf": round(rrf(h), 4),
            "IN": "YES" if h in final else "",
            "file": uri.split("://", 1)[-1].split("@")[0].removeprefix("github.com/branow/").removeprefix("hrono/"),
            "struct": sp,
        })
    df = pd.DataFrame(rows_out)
    n_sum = sum(1 for h in final if p_kind[h] == "node")
    verdict = (f"**INJECT {len(final)}** ({n_sum} from summaries)" if final
               else "**INJECT NOTHING** — consider traversing the tree")
    display(Markdown(
        f"### {prompt}\n"
        f"vector leg: mu={v_mu:.3f} sd={v_sd:.3f} floor={v_mu + Z_VEC * v_sd:.3f} · "
        f"keyword leg: mu={k_mu:.2f} sd={k_sd:.2f} floor={k_mu + Z_KW * k_sd:.2f} · "
        f"Z_VEC={Z_VEC} Z_KW={Z_KW}+support{Z_KW_SUPPORT} → {verdict}"))
    display(df.style.apply(
        lambda r: ["background-color: " + ("#e3f2fd" if r["kind"] == "SUMMARY" else "#e8f5e9")
                   if r["IN"] == "YES" else ""] * len(r), axis=1)
        .format(na_rep="—", precision=3).hide(axis="index"))

### What is the overall architecture of the webphone application?
vector leg: mu=0.774 sd=0.027 floor=0.868 · keyword leg: mu=2.95 sd=3.59 floor=24.48 · Z_VEC=3.5 Z_KW=6.0+support3.0 → **INJECT 10** (2 from summaries)

kind,cosine,z_vec,vQ,bm25,z_kw,kQ,rrf,IN,file,struct
source,0.871,3.620,,17.500,4.070,,0.015,,webphone/README.md,sp:/Web-Based%20Softphone%20System/1.%20Project%20Overview/.paragraph[1]
SUMMARY,0.870,3.590,,18.200,4.240,,0.014,,nodes/42/logging-system-for-the-webphone-application,sp:/Logging%20System%20for%20the%20WebPhone%20Application/.paragraph
source,0.879,3.920,Y,12.300,2.600,,0.014,YES,webphone/README.md,sp:/Web-Based%20Softphone%20System/3.%20Technical%20Architecture/High-Level%20System%20Overview
source,0.890,4.330,Y,0.000,-0.820,,0.011,YES,webphone/server/src/main/java/com/scisbo/webphone/WebphoneApplication.java,
source,0.883,4.070,Y,3.500,0.160,,0.011,YES,webphone/server/pom.xml,sp:/project/.content/description sp:/project/.content/properties
SUMMARY,0.880,3.960,Y,11.200,2.290,,0.011,YES,nodes/47/webphone-client-frontend,sp:/WebPhone%20Client%20Frontend/.paragraph
SUMMARY,0.878,3.870,Y,6.600,1.030,,0.011,YES,nodes/37/webphone-project-documentation-and-configuration,sp:/Webphone%20Project%20Documentation%20and%20Configuration/.paragraph
source,0.874,3.740,Y,2.200,-0.220,,0.011,YES,webphone/README.md,sp:/Web-Based%20Softphone%20System/2.%20Features%20and%20Functionalities/.paragraph
source,0.874,3.730,Y,—,—,,0.010,YES,webphone/README.md,sp:/Web-Based%20Softphone%20System/1.%20Project%20Overview/.paragraph
source,0.874,3.730,Y,3.200,0.070,,0.010,YES,webphone/client/README.md,


### What conventions do the webphone React components follow across the codebase?
vector leg: mu=0.772 sd=0.031 floor=0.881 · keyword leg: mu=2.91 sd=2.94 floor=20.52 · Z_VEC=3.5 Z_KW=6.0+support3.0 → **INJECT 6** (6 from summaries)

kind,cosine,z_vec,vQ,bm25,z_kw,kQ,rrf,IN,file,struct
SUMMARY,0.899,4.080,Y,20.000,5.840,,0.016,YES,nodes/46/webphone-client-react-ui-codebase-overview,sp:/Webphone%20Client%20%E2%80%94%20React%20UI%20Codebase%20Overview/.paragraph
SUMMARY,0.902,4.150,Y,12.100,3.130,,0.015,YES,nodes/22/webphone-client-react-ui-component-library,sp:/Webphone%20Client%20React%20UI%20Component%20Library/.paragraph
SUMMARY,0.901,4.140,Y,9.900,2.400,,0.015,YES,nodes/29/react-button-components-webphone-client-ui-library,sp:/React%20Button%20Components%20%E2%80%93%20Webphone%20Client%20UI%20Library/.paragraph
SUMMARY,0.890,3.770,Y,10.100,2.440,,0.014,YES,nodes/15/webphone-client-navigation-settings-ui-components,sp:/Webphone%20Client%20Navigation%20%26%20Settings%20UI%20Components/.paragraph
SUMMARY,0.892,3.840,Y,9.500,2.260,,0.014,YES,nodes/34/webphone-client-ui-components,sp:/Webphone%20Client%20UI%20Components/.paragraph
SUMMARY,0.880,3.460,,10.000,2.410,,0.014,,nodes/48/a-two-application-spring-boot-react-ecosystem-webphone-and-i,sp:/A%20Two-Application%20Spring%20Boot%20%2F%20React%20Ecosystem%3A%20WebPhone%20and%20Its%20Sibling%20Footba/Frontend%20and%20Tooling%20Conventions
SUMMARY,0.896,3.970,Y,8.500,1.910,,0.013,YES,nodes/39/webphone-call-ui-components,sp:/WebPhone%20Call%20UI%20Components/.paragraph
SUMMARY,0.877,3.370,,9.800,2.350,,0.013,,nodes/40/webphone-client-react-components-account-contact-and-call-ma,sp:/WebPhone%20Client%20React%20Components%20%E2%80%94%20Account%2C%20Contact%2C%20and%20Call%20Management%20UI/.atx_heading sp:/WebPhone%20Client%20React%20Components%20%E2%80%94%20Account%2C%20Contact%2C%20and%20Call%20Management%20UI/.paragraph
SUMMARY,0.854,2.630,,14.600,3.990,,0.013,,nodes/46/webphone-client-react-ui-codebase-overview,sp:/Webphone%20Client%20%E2%80%94%20React%20UI%20Codebase%20Overview/Cross-Cutting%20Patterns%20and%20Hazards/.paragraph[1]
SUMMARY,0.850,2.500,,14.300,3.870,,0.012,,nodes/46/webphone-client-react-ui-codebase-overview,sp:/Webphone%20Client%20%E2%80%94%20React%20UI%20Codebase%20Overview/Cross-Cutting%20Patterns%20and%20Hazards/.list/.list_item[3]


### What are the known cross-cutting hazards and risky patterns in the webphone frontend?
vector leg: mu=0.757 sd=0.024 floor=0.841 · keyword leg: mu=2.95 sd=4.09 floor=27.51 · Z_VEC=3.5 Z_KW=6.0+support3.0 → **INJECT 2** (2 from summaries)

...

kind,cosine,z_vec,vQ,bm25,z_kw,kQ,rrf,IN,file,struct
SUMMARY,0.841,3.340,,12.400,2.720,,0.014,,nodes/40/webphone-client-react-components-account-contact-and-call-ma,sp:/WebPhone%20Client%20React%20Components%20%E2%80%94%20Account%2C%20Contact%2C%20and%20Call%20Management%20UI/Cautions
source,0.838,3.210,,11.800,2.510,,0.013,,webphone/client/src/components/contact/photo/Photo.tsx,sp:/.lexical_declaration[2]
source,0.851,3.730,Y,6.000,0.470,,0.011,YES,webphone/client/src/components/contact/form/PhotoForm.tsx,sp:/.lexical_declaration[1]/PhotoForm/.arrow_function/.statement_block
source,0.842,3.380,,4.100,-0.220,,0.011,,webphone/server/src/main/java/com/scisbo/webphone/models/Contact.java,
source,0.838,3.230,,3.200,-0.510,,0.011,,webphone/client/src/components/contact/form/PhotoForm.tsx,sp:/.import_statement[5] sp:/.import_statement[6]
source,0.838,3.220,,—,—,,0.011,,webphone/server/src/test/java/com/scisbo/webphone/services/ContactServiceTest.java,sp:/.import_declaration[32] sp:/.import_declaration[33]
source,0.837,3.170,,2.500,-0.780,,0.010,,webphone/client/src/components/contact/ContactNumber.tsx,sp:/.import_statement[6] sp:/.import_statement[7] sp:/.import_statement[8]
SUMMARY,0.837,3.170,,9.600,1.740,,0.010,,nodes/32/summary-webphone-application-account-contact-dtos,sp:/Summary%3A%20WebPhone%20Application%20Account%20%26%20Contact%20DTOs/Cautions
source,0.836,3.140,,2.900,-0.630,,0.010,,webphone/client/src/components/contact/ContactPreview.tsx,sp:/.lexical_declaration
source,0.835,3.120,,—,—,,0.010,,webphone/server/src/main/java/com/scisbo/webphone/services/ContactService.java,sp:/.import_declaration[21] sp:/.import_declaration[22]


### How is payment processing implemented and which payment provider API is used?
vector leg: mu=0.737 sd=0.022 floor=0.813 · keyword leg: mu=4.49 sd=2.82 floor=21.39 · Z_VEC=3.5 Z_KW=6.0+support3.0 → **INJECT NOTHING** — consider traversing the tree

kind,cosine,z_vec,vQ,bm25,z_kw,kQ,rrf,IN,file,struct
SUMMARY,0.801,2.950,,10.600,2.170,,0.014,,nodes/20/webphone-client-react-context-hooks-and-api-services,sp:/WebPhone%20Client%3A%20React%20Context%2C%20Hooks%2C%20and%20API%20Services/.paragraph
source,0.795,2.670,,28.800,8.640,,0.013,,webphone/LICENSE,sp:/.~blank-line[89]/.~line[4] sp:/.~blank-line[89]/.~line[5]
SUMMARY,0.809,3.290,,2.800,-0.600,,0.011,,nodes/20/webphone-client-react-context-hooks-and-api-services,sp:/WebPhone%20Client%3A%20React%20Context%2C%20Hooks%2C%20and%20API%20Services/Account%20Management/.list[1]
SUMMARY,0.803,3.060,,7.900,1.200,,0.011,,nodes/19/webphone-voip-client-react-frontend,sp:/WebPhone%20VoIP%20Client%20%E2%80%94%20React%20Frontend/API%20%26%20Service%20Layer/.paragraph sp:/WebPhone%20VoIP%20Client%20%E2%80%94%20React%20Frontend/API%20%26%20Service%20Layer/.list
SUMMARY,0.803,3.050,,3.800,-0.250,,0.011,,nodes/20/webphone-client-react-context-hooks-and-api-services,sp:/WebPhone%20Client%3A%20React%20Context%2C%20Hooks%2C%20and%20API%20Services/Providers%20%26%20Rendering
source,0.803,3.020,,—,—,,0.011,,webphone/client/src/hooks/useSip.ts,sp:/Return
SUMMARY,0.802,3.000,,5.700,0.430,,0.011,,nodes/23/account-management-module-spring-boot-rest-api,sp:/Account%20Management%20Module%20%E2%80%94%20Spring%20Boot%20REST%20API/Mapping%20Layer
SUMMARY,0.801,2.940,,5.200,0.240,,0.010,,nodes/23/account-management-module-spring-boot-rest-api,sp:/Account%20Management%20Module%20%E2%80%94%20Spring%20Boot%20REST%20API/Service%20Layer%20%26%20Business%20Logic
source,0.800,2.900,,—,—,,0.010,,webphone/client/src/pages/account/update/UpdateAccountFormPage.tsx,sp:/.import_statement[5]
source,0.800,2.890,,2.200,-0.800,,0.010,,webphone/client/src/services/accounts.ts,sp:/.export_statement[7]


### Where is the Kafka topic configuration and how are events consumed?
vector leg: mu=0.725 sd=0.022 floor=0.801 · keyword leg: mu=4.62 sd=2.65 floor=20.52 · Z_VEC=3.5 Z_KW=6.0+support3.0 → **INJECT NOTHING** — consider traversing the tree

kind,cosine,z_vec,vQ,bm25,z_kw,kQ,rrf,IN,file,struct
source,0.796,3.230,,1.600,-1.120,,0.011,,webphone/server/src/main/java/com/scisbo/webphone/log/config/LoggingConfiguration.java,
SUMMARY,0.791,3.010,,8.400,1.420,,0.011,,nodes/6/logging-infrastructure-for-webphone-application,sp:/Logging%20Infrastructure%20for%20WebPhone%20Application/Server-Side%20Configuration%20%28Spring%20Beans%29
source,0.788,2.870,,—,—,,0.011,,webphone/server/src/main/java/com/scisbo/webphone/log/aspect/LoggingAspect.java,sp:/LoggingAspect/.modifiers sp:/LoggingAspect/.class sp:/LoggingAspect/.identifier
source,0.787,2.830,,2.400,-0.830,,0.011,,webphone/server/src/main/java/com/scisbo/webphone/log/aspect/LoggingAspect.java,sp:/.block_comment
source,0.786,2.770,,—,—,,0.011,,webphone/server/src/test/java/com/scisbo/webphone/log/aspect/LoggingAspectTest.java,sp:/.import_declaration[13] sp:/.import_declaration[14] sp:/.import_declaration[15]
SUMMARY,0.784,2.690,,8.800,1.590,,0.011,,nodes/6/logging-infrastructure-for-webphone-application,sp:/Logging%20Infrastructure%20for%20WebPhone%20Application/Caution
source,0.784,2.680,,—,—,,0.010,,webphone/client/src/components/sip/SipEventsHandler.tsx,
source,0.781,2.560,,—,—,,0.010,,webphone/server/src/main/java/com/scisbo/webphone/log/aspect/LoggingAspect.java,sp:/.import_declaration[9] sp:/.import_declaration[10]
SUMMARY,0.781,2.550,,9.400,1.800,,0.010,,nodes/37/webphone-project-documentation-and-configuration,sp:/Webphone%20Project%20Documentation%20and%20Configuration/Configuration%20and%20Environment
source,0.781,2.550,,3.600,-0.390,,0.010,,webphone/server/src/main/java/com/scisbo/webphone/config/LoggingConfig.java,sp:/LoggingConfig


## Observations

(To fill after review: do architectural prompts inject summary paragraphs; do detail
prompts regress; do controls still fail safe; does the node-URI scheme hold up.)